# Module 7 capstone, metrology twin: virtual metrology that stays in sync

**Digital Twins for Semiconductor Manufacturing · Module 7 · about 2 hours** · notebook version 2026-10-25

This is the **metrology twin** track of the capstone. You join two prepared components into one twin that predicts the oxide thickness of every product wafer of the course's furnace, and test it the way a twin should be tested before anyone relies on it:

- the **hybrid virtual-metrology model** of Module 4 predicts each wafer from the run's trace summary: the Deal-Grove physics prediction plus a learned ridge correction;
- the **offset estimate** of Module 5, a Kalman filter, keeps those predictions in sync, using one sampled wafer per run (slot 65) that is still measured.

The thermocouples age in production, so a model trained once slowly drifts away from the furnace; the sampled wafer is what brings it back. You write two short functions that connect the components, one function that turns a test result into a verdict, and the **twin data sheet**: what the twin is for, what it uses, how it performed, where it must not be used, one risk, and what to do when confidence is low.

The test plan is fixed before any test runs:

| test | data | pass when |
|---|---|---|
| held-out | production runs 41 to 100, every wafer predicted before its run's sampled wafer is used | mean absolute error (MAE) at most 0.5 nm |
| baseline | the mean of the last 5 sampled wafers, on the same runs | (for comparison) |
| challenging | runs 101 to 115, after the center thermocouple was replaced | MAE at most 0.5 nm |

Steps 1 to 8 use the **course example furnace (KEY 2026)**, so you can compare your numbers with the Module 7 page. Step 9 repeats the tests on your own furnace, with the KEY from Question 1 of the **Module 7 project quiz for the metrology twin**. All data are **simulated**. Run each code cell with **Shift + Enter**, in order.

## Step 0: start Python

The first time, Python can take up to a minute to start, and loading scikit-learn adds a little more. The cell prints the versions of the libraries and of the course files next to this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy             # the course files use SciPy: importing it here loads it in the browser
import sklearn
import furnace, vm_tools, capstone, metrology_twin
from vm_tools import FEATURES, physics_prediction
from capstone import mae_ci, blank_datasheet, show_options, show_datasheet
from metrology_twin import (vm_production, train_correction, kalman_update, run_twin, per_run_mae, plot_log,
                            TOLERANCE_NM, HELDOUT_RUNS, CHALLENGING_RUNS, SAMPLED_SLOT)

EXAMPLE_KEY = 2026
TOLERANCE = TOLERANCE_NM
print("Notebook version 2026-10-25")
print("NumPy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| furnace.py", furnace.__version__,
      "| vm_tools.py", vm_tools.__version__, "| capstone.py", capstone.__version__, "| metrology_twin.py", metrology_twin.__version__)

## Step 1: the prepared components

A twin in a fab is rarely written from scratch: it is assembled from models and tools that were each built and tested on their own. This track supplies these components:

| component | from | what it does |
|---|---|---|
| `vm_production(key)` | new data | 115 runs of recipe DRYOX-90: five wafers per run with their trace summary and thickness |
| `physics_prediction(table)` | Module 4 | the Deal-Grove thickness at each wafer's center-zone reading and oxidation time |
| `train_correction(train)` | Module 4 | the scaler-and-ridge pipeline fitted to the residual, measured minus physics, with run-blocked CV |
| `kalman_update(d, P, y, Q, R)` | Module 5 | one update of the offset estimate |
| `run_twin(key, vm_predict, offset_observation)` | new | runs 41 to 115 with the twin in the loop, using your two connections |
| `per_run_mae(log, column, runs)`, `mae_ci(errors)` | Module 6 | the mean absolute error of each run's five wafers, and the MAE with its 95 % interval |

What is missing is the **connection**: how the twin turns the two parts of the hybrid model into one prediction per wafer, and how it turns the sampled wafer's measurement into an observation of the offset. That is your part, in Step 3.

## Step 2: the data and the hybrid model

The first 40 runs, with five measured wafers each, train the learned correction of the hybrid model. In production (runs 41 on), only the sampled wafer of each run (slot 65) goes to the twin. For this validation the other four wafers were measured too, so that the twin's predictions can be checked on every wafer; a fab does this on a sample of runs when it qualifies a virtual-metrology model.

The maintenance log has one entry that matters: the furnace's center thermocouple was **replaced before run 101**.

In [ ]:
data = vm_production(EXAMPLE_KEY)
train, prod, after = data["train"], data["production"], data["after_replacement"]
print({name: f"{t['run_index'].min()} to {t['run_index'].max()} ({t['run_index'].nunique()} runs, {len(t)} wafers)" for name, t in data.items()})
model = train_correction(train)
print("ridge penalty chosen by run-blocked cross-validation:", model.named_steps["ridge"].alpha)
print(prod.head(5).to_string(index=False))

## Step 3: your task, connect the components

The hybrid prediction for the wafers of a run is the physics prediction plus the learned correction (Module 4):

vm = `physics_prediction(table)` + `model.predict(table[FEATURES])`

`run_twin()` adds the current offset estimate `d_hat` to it. After the run, the sampled wafer is measured, and its **observation of the offset** is the measured thickness minus the hybrid prediction `vm` for that wafer (without `d_hat`: the observation is of the offset itself).

**Your task:** write `vm_predict(model, table)`, which returns one prediction (nm) per row of `table`, and `offset_observation(measured_nm, vm_nm)`, which returns one number in nm. Pass only the model's inputs to it: `table[FEATURES]`.

In [ ]:
def vm_predict(model, table):
    """The hybrid virtual-metrology prediction (nm) for every wafer (row) of table."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete vm_predict() first.")


def offset_observation(measured_nm, vm_nm):
    """The offset observed on the sampled wafer: measured minus the hybrid prediction (nm)."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete offset_observation() first.")

In [ ]:
run41 = prod[prod["run_index"] == 41]
print(np.round(vm_predict(model, run41), 2), "  (five predictions, one per wafer of run 41)")
print(np.round(run41["thickness_nm"].to_numpy(), 2), "  (what the five wafers measured)")
print(offset_observation(55.40, 55.10), "  (should be 0.3, up to rounding)")

## Step 4: verify the connected twin

**Verification and validation are different questions.** Verification asks whether the code computes what the model says. Validation asks whether the model's answers are close enough to the real system for the use you intend. (These are the definitions the National Academies use for digital twins, from the National Research Council's 2012 report.) Verification comes first: a validation result means little if the code does not do what you think it does. The tests below compare the connected twin with answers you can work out exactly.

Four tests: (1) your prediction minus the physics prediction is exactly the learned correction; (2) if the sampled wafer measures the hybrid-model prediction before offset correction (`vm`, without `d_hat`) plus 0.37 nm, the observed offset is 0.37 nm; (3) on the training runs the residual, measured minus prediction, averages to zero, as a ridge fit with an intercept must; (4) there is one prediction per wafer.

In [ ]:
vm_train = np.asarray(vm_predict(model, train), dtype=float)
ok1 = np.allclose(vm_train - physics_prediction(train), model.predict(train[FEATURES]))
ok2 = abs(offset_observation(float(vm_train[0]) + 0.37, float(vm_train[0])) - 0.37) < 1e-9
ok3 = abs(np.mean(train["thickness_nm"].to_numpy() - vm_train)) < 1e-6
ok4 = vm_train.shape == (len(train),)
for i, ok in enumerate((ok1, ok2, ok3, ok4), 1):
    print(f"verification test {i}:", "ok" if ok else "FAILED: check Step 3")

## Step 5: validate on held-out runs

Now the twin runs production: runs 41 to 100, which it has never seen. For each run it predicts all five wafers, then the sampled wafer is measured and updates the offset estimate for the next run. Each run is predicted **before** its sampled wafer is used, so every error is an honest out-of-sample error.

The five wafers of a run share the run's effects, so they are not five independent tests. The test unit is the **run**: `per_run_mae()` gives the mean absolute error of each run's five wafers, and `mae_ci()` the MAE over the runs with the half-width of its 95 % interval, t(0.975, n − 1) × s / √n, as in Module 6. Even the runs are not fully independent: the offset estimate carries part of each run's error into the next runs, most of all while it catches up after a change, so the interval is approximate.

In the lower panel of the plot, each run's mean absolute error is drawn next to the tolerance line, and each test window's MAE is the thick horizontal line with its interval shaded. The verdict compares the window's MAE and its interval with the tolerance: a single run above the line does not fail the twin, and an MAE within the tolerance does not make every run acceptable.

In [ ]:
log = run_twin(EXAMPLE_KEY, vm_predict, offset_observation, model=model)
plot_log(log, title=f"Metrology twin, example furnace (KEY {EXAMPLE_KEY})")
h_mae, h_hw, h_n = mae_ci(per_run_mae(log, "error_nm", HELDOUT_RUNS))
print(f"held-out: MAE {h_mae:.3f} nm, 95 % interval +/- {h_hw:.3f} nm, over {h_n} runs (tolerance {TOLERANCE} nm)")

## Step 6: your task, `verdict()`

A test gives an error with an interval, not a single number, and the **tolerance** was set before the test, from the twin's intended use: here the mean absolute error must be at most **0.5 nm**. Compare the whole interval with the tolerance:

- **meets**: the whole interval lies at or below the tolerance, MAE + half-width ≤ tolerance;
- **misses**: the whole interval lies above the tolerance, MAE − half-width > tolerance;
- **insufficient evidence**: the interval contains the tolerance, so this test cannot tell. More test data would narrow the interval.

"Insufficient evidence" is an honest result, not a failure of the method. It is better than a "meets" that the data do not support.

The 95 % interval from `mae_ci()` is **approximate**. It treats the test units (runs or weeks) as independent, and neighbouring units can be correlated: when an error carries over from one unit to the next, the true uncertainty is larger than the computed interval, most of all after a change, when the errors move together for several units. Read a verdict close to the tolerance with that in mind.

**Your task:** write `verdict(mae, half_width, tolerance)`. Return one of the three strings exactly as written here: `"meets"`, `"misses"` or `"insufficient evidence"`.

In [ ]:
def verdict(mae, half_width, tolerance):
    """"meets", "misses" or "insufficient evidence": the 95 % interval of the MAE, mae +/- half_width, against the tolerance."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete verdict() first.")

In [ ]:
print(verdict(0.30, 0.10, 0.5), "  (should be meets: the interval runs from 0.20 to 0.40)")
print(verdict(0.45, 0.10, 0.5), "  (should be insufficient evidence: 0.35 to 0.55 contains 0.5)")
print(verdict(0.90, 0.20, 0.5), "  (should be misses: 0.70 to 1.10 lies above 0.5)")
print()
print("held-out test of the example:", verdict(h_mae, h_hw, TOLERANCE))

## Step 7: compare with the baseline

A twin should beat the simplest thing that could work. The **baseline** here predicts every wafer of a run as the mean of the last 5 sampled wafers: what an engineer could do with no model at all. `run_twin()` records it for the same runs.

If the twin did not beat its baseline, that would be a valid finding too: the simpler method would be enough.

In [ ]:
b_mae, b_hw, b_n = mae_ci(per_run_mae(log, "baseline_error_nm", HELDOUT_RUNS))
print(f"baseline (mean of the last 5 sampled wafers): MAE {b_mae:.3f} nm, +/- {b_hw:.3f} nm")
print(f"twin:                                         MAE {h_mae:.3f} nm, +/- {h_hw:.3f} nm")
print(f"the twin's error is {h_mae / b_mae:.0%} of the baseline's")

## Step 8: the challenging condition

Every validation needs one condition that is harder than normal production. Here it is the **thermocouple replacement** before run 101. The new thermocouple reads a few degrees high; the controller holds the reading at its setpoint, so the furnace runs cooler than its traces say, and the wafers come out thinner than the model predicts from those traces. The offset estimate notices it from the sampled wafers, but it was tuned for slow drift, so it takes many runs to catch up. The test uses the 15 runs after the replacement. The challenge is built in on purpose, and the notebook says so: a twin is only trusted where it has been tested.

In [ ]:
c_mae, c_hw, c_n = mae_ci(per_run_mae(log, "error_nm", CHALLENGING_RUNS))
print(f"challenging: MAE {c_mae:.3f} nm, +/- {c_hw:.3f} nm, over {c_n} runs:", verdict(c_mae, c_hw, TOLERANCE))
print("per-run MAE after the replacement:", np.round(per_run_mae(log, "error_nm", CHALLENGING_RUNS).to_numpy(), 2))

## Step 9: your own furnace

Open the **Module 7 project quiz for the metrology twin** in Canvas. Question 1 gives you a **KEY**, a whole number from 100 to 999. Type it below in place of `None` and run the cell. It repeats Steps 5 to 8 on **your furnace** with **your functions** and prints the numbers for the data sheet.

In [ ]:
KEY = None      # <- your KEY from Question 1 of the project quiz, for example KEY = 417

if KEY is None:
    print("Type your KEY in place of None, then run this cell again.")
else:
    my_log = run_twin(KEY, vm_predict, offset_observation)
    results = {"heldout": mae_ci(per_run_mae(my_log, "error_nm", HELDOUT_RUNS)),
               "challenging": mae_ci(per_run_mae(my_log, "error_nm", CHALLENGING_RUNS)),
               "baseline": mae_ci(per_run_mae(my_log, "baseline_error_nm", HELDOUT_RUNS))}
    print(f"Your furnace, KEY {KEY} (tolerance {TOLERANCE} nm):")
    for name, (m, hw, n) in results.items():
        v = verdict(m, hw, TOLERANCE) if name != "baseline" else "(for comparison)"
        print(f"  {name:12s} MAE {m:.3f} nm   +/- {hw:.3f}   n = {n:2d}   {v}")
    plot_log(my_log, title=f"Your furnace (KEY {KEY})")

## Step 10: the twin data sheet

A twin that others will use needs a label, as a dataset or a model does (datasheets for datasets, Gebru et al. 2021; model cards, Mitchell et al. 2019). The **twin data sheet** has six fields. Fields 1, 2, 5 and 6 are fixed choices, field 3 holds your numbers from Step 9, and field 4 is a list of choices. The cell below prints the choices for this track.

How to fill in field 4, the conditions where the predictions should **not** be used:

- list every condition **outside** the range the twin was calibrated and tested in: nothing supports a claim there;
- list each tested condition whose verdict was **not** "meets" (that is, "misses" or "insufficient evidence");
- do **not** list a condition that your tests included and that met the tolerance: your results support it.

Field 5 needs a risk that this twin's data path really has, and a mitigation that reduces that risk (NIST IR 8356 discusses the security and trust risks of digital twins). Field 6 is what the user should do when the twin's confidence is low.

The check cell in Step 11 reads this data sheet and compares it with your results: the numbers must match Step 9 (3 decimals), the verdicts must follow your `verdict()` rule, and the choices must be consistent with what you found.

In [ ]:
show_options("metrology")

In [ ]:
datasheet = blank_datasheet()
datasheet["intended_use"] = None           # 1. one letter, for example "A"
datasheet["data_sources"] = None           # 2. one letter
datasheet["heldout_mae"] = None            # 3. your numbers from Step 9, 3 decimals
datasheet["heldout_verdict"] = None        #    "meets", "misses" or "insufficient evidence"
datasheet["challenging_mae"] = None
datasheet["challenging_verdict"] = None
datasheet["baseline_mae"] = None
datasheet["do_not_use"] = None             # 4. a list of letters, for example ["A", "D"]
datasheet["risk_mitigation"] = None        # 5. one letter
datasheet["low_confidence_action"] = None  # 6. one letter
datasheet["note"] = ""                     # optional, not checked

show_datasheet(datasheet, "metrology", results=globals().get("results"), tolerance=TOLERANCE, key=KEY, version="2026-10-25")

## Step 11: check your work

The cell below tests your functions on cases you cannot see, recomputes the validation results for your KEY with the course's reference functions, and compares your data sheet with them. If everything passes, it prints your **verification value**, a code made from your KEY and the three numbers of your data sheet. Type it into Question 1 of the **Module 7 project quiz for the metrology twin**.

If it says **NOT PASSED**, read the message: it tells you what to fix. Fix it, run the cells you changed, then run this cell again.

In [ ]:
# The course's check code is in dt_m7_check.py, next to this notebook.
# It is kept out of the notebook so that the course's reference solutions are not on screen.
import scipy, sklearn     # the check code uses SciPy and scikit-learn: importing them here loads them in the browser
from dt_m7_check import check_metrology_twin

passed, messages, value = check_metrology_twin(vm_predict, offset_observation, verdict, datasheet, KEY)
for line_text in messages:
    print(line_text)
if passed:
    print()
    print(f"Verification value for KEY = {KEY}: {value:.3f}")
    print("Type this number into Question 1 of the Module 7 project quiz for the metrology twin.")

## What the data sheet says about the twin

You built a twin from parts that each worked on their own, verified that the connected code does what the model says, and validated it on data it had not seen, against a tolerance set before the test, next to a baseline and under one challenging condition. The data sheet records the result for whoever uses the twin next: what it is for, what it uses, how well it did, where it must not be used, one risk, and what to do when its confidence is low.

**What the metrology twin showed.** Kept in sync by one sampled wafer per run, the hybrid model predicts the unmeasured wafers to within its tolerance while the thermocouples age, far better than the recent average of sampled wafers. After a thermocouple replacement it is wrong for many runs, because the traces it reads have changed meaning: that condition goes on the data sheet, and a fab would measure every wafer until the twin has caught up again.

**Validation is not done once.** The physical system keeps changing: thermocouples age, recipes change, tools are replaced. The National Academies' 2024 report on digital twins concludes that their verification, validation and uncertainty quantification must be a continual process, revisited as the physical system, the data and the use change. A data sheet carries a date for that reason.

**Claim only what the data support.** On simulated data, a result shows how well the twin recovers the course's simulator. It does not validate a twin of a real furnace or a real line.

To keep a copy of this notebook, choose **File > Download**.